<a href="https://colab.research.google.com/github/apmontesp/Landslides_-Applied-ML-Course/blob/main/notebooks/02_preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Notebook 02: Preprocesamiento y Análisis de Datos (Landslide4Sense)
**Proyecto de Maestría** | Universidad EAFIT

Este notebook gestiona la carga de datos desde Google Drive, realiza un análisis exploratorio de las bandas multiespectrales y prepara los datos para el entrenamiento.

## 1. Configuración del Entorno y Drive
Montamos la unidad de Drive para acceder a los archivos `.h5` de forma permanente.

In [ ]:
from google.colab import drive
import os
import h5py
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# 1.1 Montar Drive
drive.mount('/content/drive')

# 1.2 Configuración de Rutas Dinámicas
# Buscamos la carpeta Landslide4Sense en tu unidad
posibles_rutas = list(Path('/content/drive/MyDrive').glob('**/Landslide4Sense/TrainData/img'))

if posibles_rutas:
    DATA_ROOT = posibles_rutas[0].parent.parent
    print(f"✅ Ruta detectada: {DATA_ROOT}")
    
    img_list = sorted(list((DATA_ROOT / "img").glob("*.h5")))
    mask_list = sorted(list((DATA_ROOT / "mask").glob("*.h5")))
    
    print(f"Muestras de entrenamiento encontradas: {len(img_list)}")
else:
    print("❌ No se encontró la estructura de carpetas en Drive.")
    print("Asegúrate de que la carpeta se llame 'Landslide4Sense' y contenga 'TrainData'.")

## 2. Funciones de Lectura y Visualización
Creamos funciones para leer los archivos H5 y visualizar las 14 bandas (Sentinel-2, DEM y Slope).

In [ ]:
def load_h5_data(file_path):
    """Lee el primer dataset de un archivo .h5"""
    with h5py.File(file_path, 'r') as f:
        return np.array(f[list(f.keys())[0]])

def plot_sample(idx):
    img = load_h5_data(img_list[idx])
    mask = load_h5_data(mask_list[idx])
    
    # Composición RGB (B4, B3, B2) con normalización para visualización
    rgb = img[:, :, [3, 2, 1]]
    rgb = (rgb - rgb.min()) / (rgb.max() - rgb.min() + 1e-8)
    
    # Extraer Pendiente (Banda 13) y DEM (Banda 14)
    slope = img[:, :, 12]
    dem = img[:, :, 13]
    
    fig, axes = plt.subplots(1, 4, figsize=(20, 5))
    axes[0].imshow(rgb); axes[0].set_title("Sentinel-2 RGB"); axes[0].axis('off')
    axes[1].imshow(slope, cmap='terrain'); axes[1].set_title("Pendiente (Slope)"); axes[1].axis('off')
    axes[2].imshow(dem, cmap='gist_earth'); axes[2].set_title("Elevación (DEM)"); axes[2].axis('off')
    axes[3].imshow(mask, cmap='viridis'); axes[3].set_title("Máscara (Deslizamiento)"); axes[3].axis('off')
    plt.show()

# Visualizar una muestra aleatoria
if len(img_list) > 0:
    plot_sample(np.random.randint(0, len(img_list)))
else:
    print("No hay datos para mostrar.")

## 3. Análisis de Distribución de Clases
Calculamos la proporción de píxeles con deslizamientos frente a los que no tienen, para entender el desbalance de clases.

In [ ]:
def calculate_balance(n_samples=100):
    counts = {0: 0, 1: 0}
    for i in range(min(n_samples, len(mask_list))):
        m = load_h5_data(mask_list[i])
        u, c = np.unique(m, return_counts=True)
        for val, count in zip(u, c):
            counts[val] += count
    
    total = counts[0] + counts[1]
    print(f"Análisis de {n_samples} máscaras:")
    print(f"Píxeles sin deslizamiento (Clase 0): {counts[0]} ({counts[0]/total:.2%})")
    print(f"Píxeles con deslizamiento (Clase 1): {counts[1]} ({counts[1]/total:.2%})")

calculate_balance()